## Figure 3.3 — Topic distribution in the image dataset
Horizontal bar chart of 10 animal categories (source: `data/image-data/`).  
Bars sharing a colour belong to an overlapping semantic cluster.

In [ ]:
import os
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
IMAGE_DIR = '../data/image-data'

counts = {
    cat: len(os.listdir(os.path.join(IMAGE_DIR, cat)))
    for cat in sorted(os.listdir(IMAGE_DIR))
    if os.path.isdir(os.path.join(IMAGE_DIR, cat))
}
counts = pd.Series(counts)
print(f'Categories: {len(counts)}  |  Total images: {counts.sum()}')
print(counts.to_string())

In [ ]:
# ── Semantic clusters (overlapping pairs / groups) ────────────────────────────
# Validated palette slots (palette.md):
# blue #2a78d6 · orange #eb6834 · aqua #1baf7a · yellow #eda100
# magenta #e87ba4 · green #008300 · violet #6250d6 · red #e34948

CLUSTERS = {
    'Pets':          (['cat', 'dog'], '#2a78d6'),          # slot 1 blue
    'Farm animals':  (['cow', 'horse', 'sheep',
                       'chicken'], '#eb6834'),             # slot 2 orange
    'Arthropods':    (['butterfly', 'spider'], '#1baf7a'), # slot 3 aqua
    'Wild animals':  (['elephant', 'squirrel'], '#eda100'),# slot 4 yellow
}

cat_to_cluster = {cat: name for name, (cats, _) in CLUSTERS.items() for cat in cats}
cat_to_color   = {cat: color for _, (cats, color) in CLUSTERS.items() for cat in cats}
cluster_order  = list(CLUSTERS.keys())

In [ ]:
# ── Sort bars: by cluster order first, then alphabetically within cluster ─────
records = [
    {'category': cat, 'count': cnt,
     'cluster': cat_to_cluster[cat], 'color': cat_to_color[cat]}
    for cat, cnt in counts.items()
]
plot_df = (
    pd.DataFrame(records)
      .assign(cluster_rank=lambda d: d['cluster'].map(
          {name: i for i, name in enumerate(cluster_order)}))
      .sort_values(['cluster_rank', 'category'], ascending=[True, True])
      .reset_index(drop=True)
)

# ── Chart constants ───────────────────────────────────────────────────────────
SURFACE   = '#fcfcfb'
INK       = '#0b0b0b'
INK_MED   = '#52514e'
INK_MUTED = '#898781'
GRID      = '#e1e0d9'
BAR_H     = 0.55

n = len(plot_df)
fig_h = max(4, n * 0.42)

fig, ax = plt.subplots(figsize=(8, fig_h), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

# ── Bars ──────────────────────────────────────────────────────────────────────
for i, row in plot_df.iterrows():
    ax.barh(i, row['count'], height=BAR_H, color=row['color'], linewidth=0)

# ── Gridlines ─────────────────────────────────────────────────────────────────
ax.set_axisbelow(True)
ax.xaxis.grid(True, color=GRID, linewidth=1, linestyle='solid')
ax.yaxis.grid(False)

# ── Axes ──────────────────────────────────────────────────────────────────────
ax.set_yticks(range(n))
ax.set_yticklabels(plot_df['category'].str.title(), fontsize=10, color=INK)
ax.set_xlim(0, counts.max() * 1.18)
ax.set_xticks([0, 100, 200, 300, 400, 500])
ax.tick_params(axis='x', colors=INK_MUTED, labelsize=9)
ax.tick_params(axis='y', length=0)

for spine in ['top', 'right', 'bottom']:
    ax.spines[spine].set_visible(False)
ax.spines['left'].set_color(GRID)

# ── "n = 550" annotation ──────────────────────────────────────────────────────
ax.axvline(550, color=GRID, linewidth=1, linestyle='solid', zorder=0)
ax.text(555, 0, 'n = 550\nper category', fontsize=8, color=INK_MUTED, va='top')

# ── Legend ────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(facecolor=color, label=name, linewidth=0)
    for name, (_, color) in CLUSTERS.items()
]
ax.legend(
    handles=legend_handles,
    title='Cluster',
    title_fontsize=8,
    fontsize=8,
    loc='lower right',
    frameon=True,
    framealpha=0.95,
    edgecolor=GRID,
    labelcolor=INK,
)

ax.invert_yaxis()

# ── Title & subtitle ──────────────────────────────────────────────────────────
plt.tight_layout()
fig.subplots_adjust(top=0.84)
fig.text(0.02, 0.96, 'Topic distribution — image dataset',
         fontsize=12, fontweight='bold', color=INK, va='bottom')
fig.text(0.02, 0.90,
         'Colour encodes semantic cluster; bars sharing a colour may overlap in labelling tasks.',
         fontsize=8.5, color=INK_MED, va='bottom')

plt.savefig('../docs/fig3_3_image_category_distribution.png',
            dpi=150, bbox_inches='tight', facecolor=SURFACE)
plt.show()
print('Saved → docs/fig3_3_image_category_distribution.png')